# Fine-tune GN-TRVN on Kaggle (2× T4)

Run the BamiBERT, bert-tiny-stage2, and bert-base-stage2 configs in separate, sequential cells. Each training process uses both T4 GPUs through `torchrun`. The configs keep their existing three epochs, Matryoshka dimensions, CachedMNRL, validation, checkpoints, and retrieval evaluation.

In Kaggle, select **GPU T4 x2** and enable **Internet**. If this is a new Kaggle notebook, first save a version of the mining notebook and add its output as an input dataset. The old notebook's `/kaggle/working` directory is not shared automatically with a new session. The input must contain `train.parquet` with one `query, positive, negative` triplet per row. Its row count depends on how many negatives pass mining. Alternatively, set `DATA_PARQUET` to its exact path, or make the same dataset available at `HUB_DATASET_ID`.

Training outputs are written under `/kaggle/working/legal-st/outputs/`. Save a Kaggle notebook version to retain them. Run each model cell in order; a successful model is skipped on rerun in the same workspace.

In [ ]:
# Parameters
from pathlib import Path

REPO_URL = "https://github.com/QuangDiy/legal-st"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/legal-st")
DATA_PARQUET = None  # e.g. /kaggle/input/my-mining-output/data/gn-trvn-hard-negatives/train.parquet
HUB_DATASET_ID = "QuangDuy/gn-trvn-hard-negatives"  # used only if no local parquet is found
RUN_MODELS = {"bami", "tiny", "base"}

## Install dependencies and get the training code

Uses Kaggle's installed CUDA PyTorch. Do not install FlashAttention-2 on T4; the existing configs resolve to fp16 and SDPA. Set `HF_TOKEN` in the code cell below, or leave it as `None` to use the environment variable or Kaggle secret. Public datasets need no token.

In [ ]:
import os
import subprocess
import sys

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("This notebook is intended for Kaggle.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "sentence-transformers==5.1.2", "transformers==4.57.1",
     "datasets>=2.19,<4", "huggingface-hub>=0.29,<1.0",
     "accelerate>=0.26", "pyyaml>=6", "rank-bm25>=0.2.2",
     "scikit-learn>=1.4", "tabulate>=0.9", "pyarrow"],
    check=True,
)

from huggingface_hub import login

HF_TOKEN = None  # Optional: set a token string here, or use an environment variable/secret.
hf_token = HF_TOKEN or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif not (REPO_DIR / "scripts/train_embedding.py").is_file():
    raise RuntimeError(f"Existing directory is not the legal-st checkout: {REPO_DIR}")

subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)

## Locate and verify the mined dataset

The training configs expect a local folder containing `train.parquet`. This cell copies the mined file into that folder. It verifies the triplet columns and requires at least one row.

In [ ]:
import shutil

import pyarrow.parquet as pq
import torch

if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Expected Kaggle T4 x2, found {torch.cuda.device_count()} CUDA GPUs")
for gpu_index in range(2):
    print(f"GPU {gpu_index}: {torch.cuda.get_device_name(gpu_index)}")

target = REPO_DIR / "data/gn-trvn-hard-negatives/train.parquet"
working_file = Path("/kaggle/working/data/gn-trvn-hard-negatives/train.parquet")
if DATA_PARQUET is not None:
    source = Path(DATA_PARQUET)
    if not source.is_file():
        raise FileNotFoundError(f"DATA_PARQUET does not exist: {source}")
elif working_file.is_file():
    source = working_file
else:
    attached = sorted(
        p for p in Path("/kaggle/input").rglob("train.parquet")
        if "gn-trvn" in str(p).lower()
    )
    if len(attached) > 1:
        raise RuntimeError(f"Several GN-TRVN parquet files found; set DATA_PARQUET explicitly: {attached}")
    source = attached[0] if attached else None

target.parent.mkdir(parents=True, exist_ok=True)
if source is None:
    if target.is_file():
        source = target
    else:
        print(f"No local parquet found; downloading {HUB_DATASET_ID} from Hugging Face")
        download_code = (
            "import os, sys\n"
            "from datasets import load_dataset\n"
            "dataset = load_dataset(sys.argv[1], split='train', token=os.getenv('HF_TOKEN'))\n"
            "dataset.to_parquet(sys.argv[2])\n"
        )
        download_file = target.with_suffix(".download.parquet")
        subprocess.run(
            [sys.executable, "-c", download_code, HUB_DATASET_ID, str(download_file)],
            check=True,
        )
        download_file.replace(target)
        source = target

metadata = pq.read_metadata(source)
required = {"query", "positive", "negative"}
actual = set(metadata.schema.names)
if metadata.num_rows < 1 or actual != required:
    raise ValueError(
        f"Wrong training dataset at {source}: rows={metadata.num_rows}, "
        f"columns={sorted(actual)}, expected_columns={sorted(required)}"
    )
if source.resolve() != target.resolve():
    shutil.copy2(source, target)
print(f"Training data: {target} ({metadata.num_rows:,} rows, {target.stat().st_size / 1e6:.1f} MB)")

## Train with both T4 GPUs

Each call launches a fresh two-process DDP job, waits for it to finish, and records completion. A failed job stops the notebook. The configs' final retrieval evaluations run inside each job, so they can take substantial time after training.

In [ ]:
from datetime import datetime, timezone

CONFIGS = {
    "bami": "configs/gn-trvn-bamibert.yaml",
    "tiny": "configs/gn-trvn-bert-tiny-stage2.yaml",
    "base": "configs/gn-trvn-bert-base-stage2.yaml",
}
OUTPUTS = {
    "bami": "outputs/gn-trvn-bamibert",
    "tiny": "outputs/gn-trvn-bert-tiny-stage2",
    "base": "outputs/gn-trvn-bert-base-stage2",
}
if not RUN_MODELS <= CONFIGS.keys():
    raise ValueError(f"Unknown models: {sorted(RUN_MODELS - CONFIGS.keys())}")

def run_model(name: str) -> None:
    if name not in RUN_MODELS:
        print(f"Skipping {name}: not selected")
        return
    config_path = REPO_DIR / CONFIGS[name]
    output_dir = REPO_DIR / OUTPUTS[name]
    marker = output_dir / ".kaggle_completed"
    if marker.is_file():
        print(f"Skipping completed model {name}: {output_dir}")
        return
    if not config_path.is_file():
        raise FileNotFoundError(config_path)
    if not target.is_file():
        raise FileNotFoundError(target)
    env = os.environ.copy()
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["OMP_NUM_THREADS"] = "2"
    print(f"Starting {name} with 2 GPUs and {CONFIGS[name]}", flush=True)
    subprocess.run(
        [sys.executable, "-m", "torch.distributed.run", "--standalone",
         "--nnodes=1", "--nproc_per_node=2", "scripts/train_embedding.py",
         "--config", CONFIGS[name]],
        cwd=REPO_DIR, env=env, check=True,
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
    print(f"Completed {name}: {output_dir}")

In [ ]:
run_model("bami")

In [ ]:
run_model("tiny")

In [ ]:
run_model("base")

## Result locations

Each model's exported weights and config are in its `outputs/gn-trvn-*` directory. Retrieval metrics are in `retrieval_eval/` under that directory. Save a Kaggle version after the jobs finish to keep the outputs.

In [ ]:
for name, relative_output in OUTPUTS.items():
    output_dir = REPO_DIR / relative_output
    marker = output_dir / ".kaggle_completed"
    print(f"{name}: {'complete' if marker.is_file() else 'not complete'} — {output_dir}")
    for result in sorted((output_dir / "retrieval_eval").glob("*.md")):
        print(f"  {result}")